# Quickstart — the Solar-UniBench data pipeline

This notebook shows how to turn a piece of **raw** solar data into a
**standardized Solar-UniBench package** using `BaseSolarPipeline` and a
dataset adapter. It runs entirely on the shipped 1-day Folsom sample
(`data/sample/`), so it needs no full data mirror and finishes in about
a minute.

The same classes power the CLI entry points:
```bash
python scripts/build_dataset.py --sample   # exactly what this notebook does
python scripts/build_dataset.py --dataset folsom   # full build from a mirror
```

In [ ]:
import shutil
import sys
import tarfile
import tempfile
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

RAW = REPO / "data" / "sample" / "raw" / "folsom_1day"
print("raw sample:", RAW, "exists:", RAW.is_dir())

## 1. Inspect the raw sample

The raw side mirrors the upstream Folsom release: one irradiance CSV
(`timeStamp, ghi, dni, dhi`) plus the sky-camera JPEGs of that day.

In [ ]:
import pandas as pd

irr = pd.read_csv(RAW / "folsom_irradiance_20160201.csv")
jpgs = sorted((RAW / "sky_images_20160201").glob("*.jpg"))
print(f"irradiance rows: {len(irr)}  (columns: {list(irr.columns)})")
print(f"sky frames:      {len(jpgs)}  (e.g. {jpgs[0].name})")
irr.head(3)

## 2. Stage the sample into the adapter's expected shape

Every adapter declares the upstream file layout it consumes
(`Folsom_irradiance.csv` + `Folsom_sky_images_<year>.tar.bz2` with
`YYYY/MM/DD/YYYYMMDD_HHMMSS.jpg` members). We stage the sample into that
shape in a temporary directory — no copying of the full mirror needed.

In [ ]:
stage = Path(tempfile.mkdtemp(prefix="folsom_nb_"))
shutil.copy(RAW / "folsom_irradiance_20160201.csv", stage / "Folsom_irradiance.csv")
with tarfile.open(stage / "Folsom_sky_images_2016.tar.bz2", "w:bz2") as tar:
    for p in jpgs:
        tar.add(p, arcname=f"2016/02/01/{p.name}")
print("staged:", sorted(x.name for x in stage.iterdir()))

## 3. Run the REAL curation pipeline

`FolsomPipeline` subclasses `BaseSolarPipeline`: it parses the raw tables,
derives clear-sky / clearness-index / quality flags via pvlib, resamples to
the canonical cadence, and packs the sky frames into a compressed Zarr
store — producing the uniform package every loader and experiment expects.

In [ ]:
from solarunibench.pipeline.base import BaseSolarPipeline
from solarunibench.pipeline.adapters.folsom import FolsomPipeline

print("FolsomPipeline is a BaseSolarPipeline:",
      issubclass(FolsomPipeline, BaseSolarPipeline))

out_dir = REPO / "data" / "sample" / "standardized_nb"
pipe = FolsomPipeline(raw_data_dir=str(stage),
                       output_dir=out_dir, dataset_name="folsom")
manifest = pipe.run_pipeline()
print("\nmanifest keys:", sorted(manifest)[:8])

## 4. Inspect the standardized package

In [ ]:
pkg = out_dir / "folsom"
ts = pd.read_parquet(pkg / "time_series.parquet")
print(f"time_series: {len(ts)} rows x {len(ts.columns)} cols")
cols = [c for c in ts.columns][:12]
print("columns (first 12):", cols)
ts.head(3)

In [ ]:
import matplotlib.pyplot as plt

d = ts.set_index("timestamp_utc").sort_index()
fig, ax = plt.subplots(figsize=(9, 2.6))
ax.plot(d.index, d["ghi_actual"], lw=0.7, color="#E69F00", label="Observed GHI")
ax.plot(d.index, d["ghi_clearsky"], lw=0.7, ls="--", color="#52514E",
        label="Clear-sky GHI (pipeline-derived)")
ax.set_ylabel(r"GHI (W m$^{-2}$)")
ax.legend(frameon=False, fontsize=8)
fig.autofmt_xdate()
plt.show()

In [ ]:
import io

import numpy as np
from PIL import Image
import zarr

g = zarr.open_group(str(pkg / "sky_images.zarr"), mode="r")
print("zarr frames:", g["images"].shape[0])
k = g["images"].shape[0] // 2
img = np.asarray(Image.open(io.BytesIO(bytes(g["images"][k]))))
print("frame", g["images_timestamps"][k], "->", img.shape)
plt.figure(figsize=(3, 3))
plt.imshow(img)
plt.axis("off");

## 5. Next steps

- Run the experiments on this package: `python scripts/run_experiments.py --sample`
- Build any full dataset from a raw mirror: `python scripts/build_dataset.py --dataset <name>`
- Reproduce paper Figure 8: see `notebooks/02_reproduce_fig8_crossover.ipynb`